In [0]:
import json
from pyspark.sql import Row
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Fact Batting

In [0]:
batting = spark.table("silver.mlb_batting_stats")

fact_batting = (
    batting
    .withColumn("stat_type", lit("batting"))
    .withColumnRenamed("at_bats", "stat_at_bats")
    # keeping a consistent set of columns; pitching-only fields will be null here
)

In [0]:
games_lookup = spark.table("silver.mlb_schedule").select("game_pk", "game_date", "home_team_id", "away_team_id")

batting_enriched = (
    fact_batting
    .join(games_lookup, on="game_pk", how="left")
    .withColumn(
        "opponent_team_id",
        when(col("team_id") == col("home_team_id"), col("away_team_id"))
        .otherwise(col("home_team_id"))
    )
)

# Sanity Check: Any fact rows that failed to find a matching game

In [0]:
missing_game_batting = batting_enriched.filter(col("game_date").isNull()).count()

print(f"Batting rows with no matching game: {missing_game_batting}")

# Writing into Gold Layer

In [0]:
batting_enriched.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("game_date") \
    .saveAsTable("gold.fact_batting_stats")